# Single-Portfolio Bootstrap Explorer

Define a portfolio, run Monte-Carlo bootstrap, and visualise metrics.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# ensure the repo root is on the path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

from engine import config as cfg
from engine.data import load_portfolio_csv, load_all_returns
from engine.runner import run_bootstrap
from engine.simulation import simulate
from engine.metrics import compute_metrics, shannon_entropy

## 1. Define Portfolio

Edit the dictionary below or load from CSV.

In [ ]:
# --- Option A: define inline ---
portfolio = {
    "MWEQ": 0.32,
    "PRAM": 0.14,
    "IWVL": 0.07,
    "IWMO": 0.06,
    "SP5A": 0.02,
    "LGAP": 0.02,
    "EXUS": 0.10,
    "GOVH": 0.27,
}

# --- Option B: load from CSV ---
# portfolio = load_portfolio_csv("sample_portfolio.csv")

print("Portfolio weights:")
for t, w in sorted(portfolio.items(), key=lambda x: -x[1]):
    print(f"  {t}: {w:.1%}")
print(f"  Total: {sum(portfolio.values()):.2%}")

# Shannon entropy
weights_arr_display = np.array([portfolio[t] for t in sorted(portfolio)])
print(f"\n  Shannon entropy (normalised): {shannon_entropy(weights_arr_display):.4f}")

## 2. Run Bootstrap Simulation

In [ ]:
N_SIM = cfg.N_SIMULATIONS
HORIZON = cfg.HORIZON_YEARS
SEED = 42

metrics = run_bootstrap(
    portfolio,
    n_sim=N_SIM,
    horizon_years=HORIZON,
    random_seed=SEED,
)

print("\n── Metrics ──")
for k, v in sorted(metrics.items()):
    print(f"  {k:42s}: {v}")

## 3. Re-generate paths for plotting

Re-run with the same seed to get the actual paths for visualisation.

In [ ]:
rng = np.random.default_rng(SEED)
weights_arr, ret_matrix = load_all_returns(portfolio, cfg.USE_AFTER_TER_RETURNS)
paths = simulate(weights_arr, ret_matrix, N_SIM, HORIZON * 12, rng)
months = np.arange(paths.shape[1])
print(f"Paths shape: {paths.shape}  (simulations × months+1)")

## 4. Plots

### 4a. Fan chart – simulated wealth paths

In [ ]:
years = months / 12

fig = go.Figure()

bands = [(5, 95, "rgba(70,130,180,0.12)"), (20, 80, "rgba(70,130,180,0.22)"), (35, 65, "rgba(70,130,180,0.35)")]
for lo, hi, color in bands:
    p_lo = np.percentile(paths, lo, axis=0)
    p_hi = np.percentile(paths, hi, axis=0)
    fig.add_trace(go.Scatter(
        x=np.concatenate([years, years[::-1]]),
        y=np.concatenate([p_hi, p_lo[::-1]]),
        fill="toself", fillcolor=color, line=dict(width=0),
        name=f"P{lo}–P{hi}", hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(x=years, y=np.median(paths, axis=0),
                          mode="lines", line=dict(color="navy", width=2), name="Median"))

fig.update_layout(title="Simulated wealth paths – fan chart",
                  xaxis_title="Years", yaxis_title="Portfolio value (start = 1)",
                  template="plotly_white", height=500)
fig.show()

### 4b. Distribution of annualised returns

In [ ]:
ann_ret = paths[:, -1] ** (1.0 / HORIZON) - 1.0

fig = go.Figure()
fig.add_trace(go.Histogram(x=ann_ret, nbinsx=60, marker_color="steelblue",
                            opacity=0.75, name="Annualised returns"))

for p in cfg.RETURN_PERCENTILES:
    v = np.percentile(ann_ret, p)
    fig.add_vline(x=v, line_dash="dash", line_color="tomato", line_width=1,
                  annotation_text=f"P{p}: {v:.1%}", annotation_position="top")

fig.update_layout(title=f"Distribution of {HORIZON}-year annualised returns ({N_SIM} sims)",
                  xaxis_title="Annualised return", yaxis_title="Count",
                  xaxis_tickformat=".0%", template="plotly_white", height=450)
fig.show()

### 4c. Annualised return percentiles – bar chart

In [ ]:
ret_keys = [k for k in sorted(metrics) if k.startswith("annualised_return_p")]
ret_labels = [k.replace("annualised_return_", "").upper() for k in ret_keys]
ret_vals = [metrics[k] for k in ret_keys]
colors = ["#d62728" if v < 0 else "#2ca02c" for v in ret_vals]

fig = go.Figure(go.Bar(
    x=ret_labels, y=ret_vals, marker_color=colors,
    text=[f"{v:.2%}" for v in ret_vals], textposition="outside",
))
fig.add_hline(y=0, line_color="grey", line_width=0.8)
fig.update_layout(title="Annualised return at each percentile",
                  yaxis_title="Annualised return", yaxis_tickformat=".0%",
                  template="plotly_white", height=450)
fig.show()

### 4d. Volatility of N-year annualised returns

In [ ]:
vol_keys = [k for k in metrics if k.startswith("volatility_")]
vol_keys.sort(key=lambda k: int(k.split("_")[1].replace("y", "")))
vol_years = [int(k.split("_")[1].replace("y", "")) for k in vol_keys]
vol_vals = [metrics[k] for k in vol_keys]

fig = go.Figure(go.Scatter(
    x=vol_years, y=vol_vals, mode="lines+markers+text",
    marker=dict(size=10, color="darkorange"),
    line=dict(color="darkorange", width=2),
    text=[f"{v:.1%}" for v in vol_vals], textposition="top center",
))
fig.update_layout(title="Std-dev of N-year annualised returns across simulations",
                  xaxis_title="Window (years)", yaxis_title="Std-dev of annualised return",
                  yaxis_tickformat=".0%", template="plotly_white", height=420)
fig.show()

### 4e. Summary table

In [ ]:
bp = cfg.BAD_PERCENTILE
summary = {
    "Simulations": N_SIM,
    "Horizon (years)": HORIZON,
    "History months used": metrics["n_months_history"],
    "Shannon entropy": f"{metrics.get('shannon_entropy', 0):.4f}",
    f"Max DD depth (worst {bp}%)": f"{metrics.get(f'max_dd_depth_p{bp}', 0):.2%}",
    f"Max DD length (worst {bp}%)": f"{metrics.get(f'max_dd_length_months_p{bp}', 0)} months",
    f"MDA (worst {bp}%)": f"{metrics.get(f'mda_months_p{bp}', 0):.2f} equiv. months",
}

df_summary = pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])
df_summary.style.set_caption("Simulation summary")